# Module 3: Mediation, Moderation, and SEM

This module explores advanced relational modeling — how variables mediate and moderate causal paths, and how Structural Equation Modeling (SEM) tests entire theoretical frameworks simultaneously.

## Section 1: Mediating Effect

### Objective
To test whether Customer Satisfaction acts as a mediating variable that transmits the effect of Brand Image on Customer Loyalty, using the Baron-Kenny steps and bootstrapping.

**Keywords:** mediation, indirect effect, Baron-Kenny steps, bootstrapping, path a, path b, path c', partial mediation, full mediation

### Statistical Perspective
**Mediation Analysis** answers *how* or *why* an independent variable (IV) influences a dependent variable (DV). A **mediating variable** ($M$) sits in the causal chain between the IV and DV, partially or fully carrying the effect.

**The Mediation Framework:**


$$IV \xrightarrow{a} M \xrightarrow{b} DV$$

- **Path $a$:** Effect of IV on the mediator $M$.
- **Path $b$:** Effect of $M$ on DV, controlling for IV.
- **Direct path ($c'$):** Remaining direct effect of IV on DV after controlling for $M$.
- **Indirect effect ($a \times b$):** The portion of IV's effect on DV transmitted *through* $M$.

**Types of Mediation:**

- **Full Mediation:** $c'$ becomes non-significant after introducing $M$; the entire effect is indirect.
- **Partial Mediation:** Both direct ($c'$) and indirect ($a \times b$) effects are significant.

**Hypothesis Tests:**

| Path | $H_0$ | $H_1$ |
|------|---------|---------|
| Path $a$ (IV → M) | $a = 0$: Brand Image has no effect on Satisfaction | $a \neq 0$: Significant effect exists |
| Path $b$ (M → DV) | $b = 0$: Satisfaction has no effect on Loyalty | $b \neq 0$: Significant effect exists |
| Direct ($c'$) | $c' = 0$: No direct IV → DV effect remains | $c' \neq 0$: Direct effect persists after mediation |
| Indirect ($a \times b$) | Indirect effect $= 0$: No mediation | Bootstrapped CI excludes zero: Mediation confirmed |

**Significance via Bootstrapping:**
The indirect effect $a \times b$ is not normally distributed. We use bootstrapping (5,000 resamples) to build 95% CIs. If the CI does **not** include zero, mediation is significant.

**Industry Application:** In digital marketing, advertising spend (IV) increases e-commerce sales (DV) *through* increased website traffic (Mediator). Understanding this mediation determines whether to invest in ad reach or site experience.

### Python Example Code


In [ ]:
#| cache: true
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.mediation import Mediation

df = pd.read_csv('gogoro_data.csv')

outcome_model  = smf.ols('Loyal1 ~ Brand1 + Sat1', data=df)
mediator_model = smf.ols('Sat1 ~ Brand1', data=df)

med    = Mediation(outcome_model, mediator_model, 'Brand1', mediator='Sat1')
result = med.fit(n_rep=500, method='bootstrap')
print(result.summary())



> **Interpreting the Output:** The summary table reports four rows. **ACME** (Average Causal Mediation Effect) is the indirect effect $a \\times b$ — if its 95% bootstrapped CI excludes zero, Customer Satisfaction significantly mediates the Brand→Loyalty path. **ADE** (Average Direct Effect) is the direct path $c'$ — a non-significant ADE alongside a significant ACME indicates full mediation. **Total Effect** ($c = c' + a \\times b$) is the unmediated effect of Brand on Loyalty. **Prop. Mediated** shows the fraction of the total effect transmitted through Satisfaction. Use the CI columns rather than p-values to judge significance, as bootstrapped CIs are more reliable for indirect effects.

### Student Task
Create `StudentName_Mediation.py` in Visual Studio Code. Use `statsmodels.stats.mediation.Mediation` to test whether `Sat1` mediates the relationship between `Func1` (IV) and `Loyal1` (DV). Fit the outcome model as `Loyal1 ~ Func1 + Sat1` and the mediator model as `Sat1 ~ Func1`. Report the ACME and whether its 95% CI excludes zero. State whether mediation is full or partial.

### Evaluation Questions
1. What is the conceptual definition of a mediating variable?
2. Describe the three causal paths in a mediation model ($a$, $b$, and $c'$).
3. What is the indirect effect and how is it calculated?
4. Why is bootstrapping preferred over traditional significance tests for indirect effects?
5. In the Gogoro scenario, what serves as the mediator between Brand Image and Loyalty?

## Section 2: Moderating Effect

### Objective
To test whether a third variable (moderator) changes the strength or direction of the relationship between an independent variable and a dependent variable, using interaction terms in regression.

**Keywords:** moderation, interaction term, mean-centering, simple slopes, boundary condition, Johnson-Neyman, conditional effect

### Statistical Perspective
**Moderation Analysis** answers *when* or *for whom* a relationship holds. A **moderating variable** ($W$) interacts with the IV to produce a different effect on the DV depending on the level of $W$.

**The Moderation Framework:**

$$DV = b_0 + b_1(IV) + b_2(W) + b_3(IV \times W) + \epsilon$$

- **$b_3$ (Interaction Term):** A significant $b_3$ ($p < 0.05$) confirms moderation — the effect of IV on DV depends on the value of $W$.
**Hypothesis Test (Interaction Term):**

- **$H_0$:** $b_3 = 0$ — the moderator does not change the strength or direction of the IV → DV relationship; the effect of Brand Image on Satisfaction is the same regardless of Usability level.
- **$H_1$:** $b_3 \neq 0$ — the effect of Brand Image on Satisfaction depends on the level of the moderating variable.

Decision rule: If $p < 0.05$ for `Brand_x_Usab`, reject $H_0$ and conclude moderation is statistically supported.

- **Mean-Centering:** Both variables are mean-centered before creating the interaction term to reduce multicollinearity.
- **Simple Slopes:** Plot the IV-DV relationship at low (−1 SD), mean, and high (+1 SD) levels of $W$ to interpret the moderation.

**Types of Moderation:**

- **Enhancing:** Higher $W$ strengthens the IV → DV relationship.
- **Buffering:** Higher $W$ weakens the IV → DV relationship.
- **Crossover:** The direction of IV → DV flips depending on $W$.

**Industry Application:** Brand Image ($IV$) may have a stronger effect on Customer Satisfaction ($DV$) among environmentally conscious consumers ($W$ = green attitude). The moderator reveals that the same marketing message resonates differently across customer segments.

### Python Example Code


In [ ]:
import pandas as pd
import statsmodels.api as sm

df = pd.read_csv('gogoro_data.csv')

df['Brand_c'] = df['Brand1'] - df['Brand1'].mean()
df['Usab_c']  = df['Usab1']  - df['Usab1'].mean()
df['Brand_x_Usab'] = df['Brand_c'] * df['Usab_c']

X = sm.add_constant(df[['Brand_c', 'Usab_c', 'Brand_x_Usab']])
y = df['Sat1']

model = sm.OLS(y, X).fit()
print(model.summary().tables[1])



> **Interpreting the Output:** The coefficient table has four rows: `const` (intercept), `Brand_c` (main effect of Brand Image), `Usab_c` (main effect of Usability), and `Brand_x_Usab` (interaction term). Focus on **`Brand_x_Usab`**: the coefficient indicates the direction of moderation (positive = amplifying effect; negative = buffering effect), and the p-value tests $H_0$: $b_3 = 0$. If $p < 0.05$, Usability significantly changes how Brand Image affects Customer Satisfaction — the moderating hypothesis is supported. The main effect coefficients (`Brand_c`, `Usab_c`) are interpreted at the mean of the other variable due to mean-centering.

### Student Task
Create `StudentName_Moderation.py` in Visual Studio Code. Test whether `Price1` moderates the relationship between `Func1` and `Sat1`. Mean-center both `Func1` and `Price1`, create the interaction term, and run OLS. Report the coefficient and p-value of the interaction term. State whether moderation is supported.

### Evaluation Questions
1. What is the conceptual definition of a moderating variable?
2. How does moderation differ from mediation?
3. Why must predictors be mean-centered before creating an interaction term?
4. If the interaction term coefficient is $b_3 = 0.22$, $p = 0.03$, what does this mean?
5. How do "simple slopes" help interpret a significant moderation effect?

## Section 3: Structural Equation Modeling

### Objective
To build a Structural Equation Model (SEM) that simultaneously tests the full measurement model and structural paths linking Brand Image, Function, Usability, Price, Customer Satisfaction, and Customer Loyalty.

**Keywords:** structural equation modeling (SEM), confirmatory factor analysis (CFA), path analysis, CFI, RMSEA, SRMR, latent variable, model fit indices

### Statistical Perspective
**Structural Equation Modeling (SEM)** integrates **confirmatory factor analysis (CFA)** and **path analysis** into a single framework. It simultaneously estimates:

1. **Measurement model:** How well observed items reflect latent constructs (loadings, AVE, reliability).
2. **Structural model:** Directional paths between latent constructs (coefficients, p-values).

**Key Components:**

- **Latent Variables:** Unobserved constructs inferred from multiple indicators (e.g., Brand Image measured by Brand1, Brand2).
- **Structural Paths ($\gamma, \beta$):** Directional coefficients between latent variables.
- **Error Terms:** Measurement and structural residuals.

**Model Fit Indices:**

| Index | Acceptable | Good |
|-------|-----------|------|
| CFI   | $> 0.90$  | $> 0.95$ |
| RMSEA | $< 0.08$  | $< 0.06$ |
| SRMR  | $< 0.08$  | $< 0.05$ |
| $\chi^2/df$ | $< 5.0$ | $< 3.0$ |

**Hypothesis Tests (Structural Paths):**

For each path coefficient $\beta$ in the structural model:

- **$H_0$:** $\beta = 0$ — the predictor construct has no significant directional effect on the outcome construct.
- **$H_1$:** $\beta \neq 0$ — a statistically significant structural relationship exists.

Decision rule: If $p < 0.05$ for a path coefficient, reject $H_0$ and retain the hypothesized causal path in the model.

**Advantage over Regression:** SEM accounts for measurement error in latent constructs by explicitly modelling each observed item as a fallible indicator of its latent variable ($x_i = \lambda_i F + \epsilon_i$). This produces more accurate path estimates and tests the entire theoretical model simultaneously. `semopy` implements full-information maximum likelihood (FIML) estimation and reports standardized fit indices (CFI, RMSEA, SRMR) required for academic publication.

**Industry Application:** Management consulting firms use SEM to model the full chain from "Employee Engagement" → "Service Quality" → "Customer Satisfaction" → "Revenue Growth" in a single validated model.

### Python Example Code


In [ ]:
import pandas as pd
import semopy

df = pd.read_csv('gogoro_data.csv')

